# Lab 2 · Exercise A — Anatomy of a fit
Same data and model as Lab 1, Exercise C. This time we call **MIGRAD, HESSE and MINOS** ourselves with `RooMinimizer`.

In [ ]:
import ROOT
import numpy as np

MATRICOLA = 0   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

In [ ]:
vals = np.loadtxt("../data/rarest_b0_decay.dat")
m = ROOT.RooRealVar("m", "m [MeV]", 5080, 5540)
data = ROOT.RooDataSet("data", "data", ROOT.RooArgSet(m))
for v in vals:
    m.setVal(v)
    data.add(ROOT.RooArgSet(m))

mean  = ROOT.RooRealVar("mean", "mean", 5280, 5200, 5350)
sigma = ROOT.RooRealVar("sigma", "sigma", 15, 5, 50)
slope = ROOT.RooRealVar("slope", "slope", -0.002, -0.02, 0)
sig   = ROOT.RooGaussian("sig", "sig", m, mean, sigma)
bkg   = ROOT.RooExponential("bkg", "bkg", m, slope)
nsig  = ROOT.RooRealVar("nsig", "signal yield", 30, 0, 166)
nbkg  = ROOT.RooRealVar("nbkg", "background yield", 130, 0, 500)
model = ROOT.RooAddPdf("model", "sig+bkg", ROOT.RooArgList(sig, bkg), ROOT.RooArgList(nsig, nbkg))
print(data.numEntries(), "events")

## A1 · NLL, MIGRAD, HESSE
Create the extended NLL, run Minuit2 MIGRAD, then HESSE. Save the result and read the status, the EDM and `covQual`.

In [ ]:
nll   = None   # TODO: model.createNLL(data, ROOT.RooFit.Extended(True))
minim = None   # TODO: ROOT.RooMinimizer(nll)

# TODO: minim.setPrintLevel(-1); minim.minimize("Minuit2", "Migrad"); minim.hesse()
res = None     # TODO: res = minim.save()

err_hesse = nsig.getError()      # keep it: MINOS will update the error object later
print("status =", res.status(), " covQual =", res.covQual(), " EDM =", res.edm())
print("nsig = %.2f +- %.2f (HESSE)" % (nsig.getVal(), err_hesse))

In [ ]:
check("minimiser objects exist", nll is not None and minim is not None and res is not None)
check("fit converged (status 0)", res.status() == 0)
check("full covariance matrix (covQual 3)", res.covQual() == 3)

## A2 · MINOS
Compute MINOS errors for `nsig` only. Compare with HESSE.

In [ ]:
# TODO: minim.minos(ROOT.RooArgSet(nsig))
lo = nsig.getErrorLo()
hi = nsig.getErrorHi()
print("MINOS: nsig = %.2f  %+.2f / %+.2f" % (nsig.getVal(), lo, hi))
print("HESSE: nsig = %.2f +- %.2f" % (nsig.getVal(), err_hesse))

In [ ]:
check("MINOS lower error is negative and upper positive", lo < 0 < hi)

## A3 · Correlations
Print the correlation matrix of the fit. Which pair is most correlated?

In [ ]:
# TODO: print the correlation matrix of res;  res.correlationMatrix().Print()
# TODO: print the correlation between "nsig" and "nbkg":  res.correlation("nsig", "nbkg")

## A4 · NLL scan, profile, and the HESSE parabola
Plot three curves of $-\ln L$ against `nsig`, shifted to zero at the minimum:
the **slice** (other parameters frozen), the **profile** (other parameters re-fitted), and the **parabola** $0.5\,((x-\hat n)/\sigma_{\rm HESSE})^2$.

In [ ]:
frame = nsig.frame(ROOT.RooFit.Range(10, 75), ROOT.RooFit.Title("-lnL vs nsig"))
frame.SetMaximum(4)
frame.SetMinimum(0)
# TODO: slice:   nll.plotOn(frame, ROOT.RooFit.ShiftToZero())
# TODO: profile: prof = nll.createProfile(ROOT.RooArgSet(nsig));  prof.plotOn(frame, ROOT.RooFit.LineColor(ROOT.kRed))

parab = ROOT.TF1("parab", "0.5*((x-[0])/[1])**2", 10, 75)
parab.SetParameters(nsig.getVal(), err_hesse)
parab.SetLineStyle(2)

cs = ROOT.TCanvas("cs", "cs", 800, 500)
frame.Draw()
parab.Draw("same")
cs.Draw()

## A5 · Contour
Draw the 1σ and 2σ contours of (`nsig`, `nbkg`) with `minim.contour(...)`.

In [ ]:
# TODO: cont = minim.contour(nsig, nbkg, 1, 2)   # returns a RooPlot
# TODO: draw it on a canvas

## Questions (2–3 lines each)
1. Are HESSE and MINOS errors on `nsig` equal? If not, which side is longer, and why?
2. Which two parameters are most correlated? Why does the physics of the model produce this?
3. At which values of `nsig` do the three curves in A4 separate most? What does this say about the likelihood?